# GAVE2 Task 1 training on Kaggle

Before running: Notebook settings (right panel) -> Accelerator: **GPU T4 x2**, Internet: **On**.
Make sure the `gave2-preliminary` and `gave2-registered` datasets are added as inputs (Add Input -> Your Datasets).

## Cell 1: clone the repo (public -- no token needed)

In [ ]:
import os

repo_url = "https://github.com/aaronjji/gave2-challenge.git"

if not os.path.exists("/kaggle/working/gave2-challenge"):
    os.system(f"git clone --recurse-submodules {repo_url} /kaggle/working/gave2-challenge")
else:
    os.system("cd /kaggle/working/gave2-challenge && git pull && git submodule update --init --recursive")

os.chdir("/kaggle/working/gave2-challenge")
print(os.getcwd())

## Cell 2: install dependencies
(torch/torchvision are preinstalled on Kaggle -- skip those.)

In [ ]:
os.system("pip install -q albumentations opencv-python-headless scikit-image scipy networkx sknw huggingface_hub safetensors")

## Cell 3: point at the uploaded dataset
Lists the input dir first so you can see the real structure if the assert below fails --
Kaggle's zip auto-extract occasionally flattens or nests differently than expected.

In [ ]:
candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary",
]
os.system("find /kaggle/input -maxdepth 4")

DATA_ROOT = None
for c in candidates:
    if os.path.exists(f"{c}/training/images"):
        DATA_ROOT = c
        break
assert DATA_ROOT is not None, f"Dataset not found in any of {candidates} -- check the find output above and set DATA_ROOT manually"
print("DATA_ROOT =", DATA_ROOT)

## Cell 4: restore checkpoints from a previous session
Kaggle sessions cap out well below the 5-fold budget, so this push spans multiple sessions. After each session finishes, **Save Version** (this auto-happens with Save & Run All), then before starting the next session: **Add Input -> Your Work -> Notebooks -> this notebook -> latest version**. This cell finds that version's `runs/` output (if attached) and restores ALL fold checkpoints from it (Task 1 fold0 + any completed Task 2 folds), so nothing retrains from scratch.

In [ ]:
import shutil
from pathlib import Path

candidates = [p for p in Path("/kaggle/input").rglob("runs") if (p / "task1").exists() or (p / "task2").exists()]
if candidates:
    prev_runs = candidates[0]
    print(f"Restoring checkpoints from previous session output: {prev_runs}")
    shutil.copytree(prev_runs, "runs", dirs_exist_ok=True)
    for f in sorted(Path("runs").rglob("*.pth")):
        print(" ", f)
else:
    print("No previous session output attached -- starting fresh (expected for session 1 only)")

## Cell 5: train one fold
Kaggle sessions cap out around 9-12h; `--max-seconds` leaves a safety margin so the run
checkpoints and exits cleanly rather than getting killed mid-step. Re-run this same cell
(with `--resume`) if you need more epochs than one session allows -- it picks up from
`runs/task1/fold{N}/latest.pth`.

In [ ]:
FOLD = 0
os.system(
    f"python -u src/train_task1.py "
    f"--fold {FOLD} --data-root {DATA_ROOT} "
    f"--epochs 80 --steps-per-epoch 50 --num-workers 2 "
    f"--checkpoint-every-epochs 2 --val-every-epochs 5 --max-seconds 30000 "
    f"--out-dir runs/task1 --resume"
)

## Cell 6: repeat for the other folds
Change `FOLD` to 1, 2, 3, 4 and re-run Cell 5 (each fold trains independently). Once
satisfied, **Save Version** to persist `runs/` as this notebook's output, then download the
checkpoints (or add this version as an input to a follow-up inference notebook/session).

## Cell 7: Task 2 -- upload registered FFA data first
Task 2 needs CFP<->FFA registration (they are NOT pre-aligned -- verified empirically). Registration was already run locally (MINIMA sp_lg, 100/100 pairs succeeded on both splits, zero fallbacks). Upload the local `data/registered/` folder (~232MB) as a **new private Kaggle Dataset** the same way you did the main one, add it as an input to this notebook, then run the cell below.

In [ ]:
reg_candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-registered/registered",
    "/kaggle/input/gave2-registered/registered",
    "/kaggle/input/gave2-registered",
]
os.system("find /kaggle/input -maxdepth 5")

FFA_ROOT = None
for c in reg_candidates:
    if os.path.exists(f"{c}/training/FFA_A"):
        FFA_ROOT = c
        break
if FFA_ROOT is None:
    print("WARNING: registered FFA dataset not found -- Task 2 cells below will skip themselves. "
          "Attach the 'GAVE2 Registered' dataset and re-run this cell if you want Task 2.")
else:
    print("FFA_ROOT =", FFA_ROOT)

## Cell 8: train Task 2, 5-fold, multiple folds per session
Warm-starts every fold from the single Task 1 fold0 checkpoint (Task 1 isn't re-trained per fold -- reuses Cell 5's checkpoint, saves ~4h/fold). Set `FOLDS_THIS_SESSION` below to which folds to train *this* session (e.g. `[0, 1]`, then next session `[2, 3]`, then `[4]`) -- already-completed folds (final.pth present) are skipped automatically, and a fold cut off mid-training resumes from its own latest.pth. `SESSION_BUDGET_SECONDS` is split dynamically across the folds in the list so the session exits cleanly instead of getting killed.

In [ ]:
import time as _time

FOLDS_THIS_SESSION = [0, 1]  # <-- change per session: e.g. [2, 3] next, [4] after that
SESSION_BUDGET_SECONDS = 28800  # ~8h, safety margin under Kaggle's ~9h GPU session cap
EPOCHS_PER_FOLD = 90  # ~90 x 158s/epoch (T4, measured from the earlier 60-epoch run) ~= 3.95h/fold

if FFA_ROOT is None:
    print("Skipping Task 2 training -- FFA_ROOT not set (see Cell 7).")
else:
    session_t0 = _time.time()
    for FOLD2 in FOLDS_THIS_SESSION:
        fold_dir = Path(f"runs/task2/fold{FOLD2}")
        if (fold_dir / "final.pth").exists():
            print(f"Fold {FOLD2} already complete (final.pth found) -- skipping")
            continue
        elapsed = _time.time() - session_t0
        remaining = SESSION_BUDGET_SECONDS - elapsed
        if remaining < 600:
            print(f"Only {remaining:.0f}s left in session budget -- stopping before fold {FOLD2}")
            break
        print(f"=== Training Task2 fold {FOLD2} (budget: {remaining:.0f}s) ===")
        os.system(
            f"python -u src/train_task2.py "
            f"--fold {FOLD2} --data-root {DATA_ROOT} --ffa-root {FFA_ROOT} "
            f"--warm-start-task1 runs/task1/fold0/latest.pth "
            f"--epochs {EPOCHS_PER_FOLD} --steps-per-epoch 50 --num-workers 2 "
            f"--checkpoint-every-epochs 2 --val-every-epochs 5 --max-seconds {remaining:.0f} "
            f"--out-dir runs/task2 --resume"
        )

## Cell 9: run inference (Task 1 single-fold, Task 2 5-fold ensemble)
Task 2 automatically ensembles across every fold checkpoint that exists yet (averages sigmoid probability maps before saving) -- safe to run after only some folds are done, it'll just ensemble fewer than 5. Both cells are safe to skip if that task has no checkpoint at all yet.

In [ ]:
# Task 1 inference -- prefer best.pth (lowest validation loss) over latest.pth
task1_ckpt = "runs/task1/fold0/best.pth" if os.path.exists("runs/task1/fold0/best.pth") else "runs/task1/fold0/latest.pth"
if not os.path.exists(task1_ckpt):
    print(f"Skipping Task 1 inference -- no checkpoint found")
else:
    print(f"Using checkpoint: {task1_ckpt}")
    os.system(
        "python -u src/predict_task1.py --task task1 "
        f"--checkpoint {task1_ckpt} "
        f"--images-dir {DATA_ROOT}/validation/images "
        f"--masks-dir {DATA_ROOT}/validation/masks "
        "--out-dir predictions/task1/validation"
    )

In [ ]:
# Task 2 inference -- ensemble across every fold checkpoint that exists so far
# (prefers each fold's best.pth over latest.pth)
task2_ckpts = []
for _fold in range(5):
    _best = Path(f"runs/task2/fold{_fold}/best.pth")
    _latest = Path(f"runs/task2/fold{_fold}/latest.pth")
    if _best.exists():
        task2_ckpts.append(str(_best))
    elif _latest.exists():
        task2_ckpts.append(str(_latest))

if FFA_ROOT is None or not task2_ckpts:
    print("Skipping Task 2 inference -- FFA_ROOT not set or no checkpoints found")
else:
    print(f"Ensembling {len(task2_ckpts)} Task2 checkpoint(s): {task2_ckpts}")
    os.system(
        "python -u src/predict_ensemble.py --task task2 "
        f"--checkpoints {' '.join(task2_ckpts)} "
        f"--images-dir {DATA_ROOT}/validation/images "
        f"--masks-dir {DATA_ROOT}/validation/masks "
        f"--ffa-dir {FFA_ROOT}/validation "
        "--out-dir predictions/task2/validation"
    )

## Cell 10: Task 3 biomarkers
Computed from whichever prediction set you want to submit against (Task 2's should be more accurate once trained, since it has the FFA signal Task 1 doesn't). Uses the placeholder heuristic optic-disc detector -- not a trained OD model yet.

In [ ]:
if os.path.exists("predictions/task2/validation"):
    AV_SOURCE = "predictions/task2/validation"
elif os.path.exists("predictions/task1/validation"):
    AV_SOURCE = "predictions/task1/validation"
else:
    AV_SOURCE = None

if AV_SOURCE is None:
    print("Skipping Task 3 -- no Task1/Task2 predictions found")
else:
    print("Computing Task3 from:", AV_SOURCE)
    os.system(
        f"python -u src/run_task3.py "
        f"--av-dir {AV_SOURCE} "
        f"--images-dir {DATA_ROOT}/validation/images "
        f"--masks-dir {DATA_ROOT}/validation/masks "
        f"--out-dir predictions/task3/validation"
    )

## Cell 11: package the submission
Set `--task1-dir`/`--task2-dir` to whichever prediction sets you actually generated above (omit a `--taskN-dir` flag entirely to leave that task out of the zip -- partial submissions are allowed).

In [ ]:
task_flags = ""
if os.path.exists("predictions/task1/validation"):
    task_flags += " --task1-dir predictions/task1/validation"
if os.path.exists("predictions/task2/validation"):
    task_flags += " --task2-dir predictions/task2/validation"
if os.path.exists("predictions/task3/validation"):
    task_flags += " --task3-dir predictions/task3/validation"

if not task_flags:
    print("Nothing to package -- no prediction outputs found")
else:
    os.system(
        "python -u src/format_submission.py --team-id aarons_team "
        + task_flags +
        " --out-zip submissions/submission.zip"
    )
    print("Download it from the file browser: /kaggle/working/gave2-challenge/submissions/submission.zip")